In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots
from figures.generate_data import load_image_point_cloud
from diffusion_geometry.classes.main import DiffusionGeometry


Create some data

In [ ]:
data1 = load_image_point_cloud("../data/data2.jpeg", n=1000, threshold=0.9, intensity_weighted=True)

plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
f = dg1.function_space.zeros()
f.coeffs[7] = -0.5
f.coeffs[2] = -1
f11 = plot_scatter_2d(data1, color = f.to_ambient(), size=4)
f11.show()

f12 = plot_quiver_2d(data1, f.grad().to_ambient(), scale = 0.015, line_width=1.2)
f12.show()

In [ ]:
data2 = load_image_point_cloud("../data/data3.jpeg", n=800, threshold=0.95, intensity_weighted=False)

plot_scatter_2d(data2, color = 'black').show()
dg2 = DiffusionGeometry.from_point_cloud(data2)


In [ ]:
# f = dg2.function(data2[:,0] + 2*data2[:,1])
f = dg2.function_space.zeros()
f.coeffs[8] = 0.5
f21 = plot_scatter_2d(data2, color = f.to_ambient())
f21.show()

f22 = plot_quiver_2d(data2, f.grad().to_ambient(), scale = 0.02, line_width=1.6)
f22.show()

In [ ]:
from generate_data import gen_3d_data
data3, _ = gen_3d_data(kind = "swiss_roll", n = 2000, noise = 0.05)
dg3 = DiffusionGeometry.from_point_cloud(data3)


camera = dict(eye=dict(x=-0.4, y=1.85, z=0.5))
plot_scatter_3d(data3, color = 'black', camera=camera).show()

In [ ]:
f = dg3.function_space.zeros()
f.coeffs[4] = -0.2
f.coeffs[6] = 1

f31 = plot_scatter_3d(data3, color = f.to_ambient(), camera = camera)
f31.show()

f32 = plot_quiver_3d(data3, f.grad().to_ambient(), scale = 6, line_width=2, arrow_scale=0.6, camera=camera)
f32.update_layout(scene_camera=camera)
f32.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}],
    # [{"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    # rows=3,
    rows=2,
    cols=2,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.0,
    row_heights=[0.3, 0.7]
)

# --- Add traces row by row ---
fig.add_traces(list(f11.data), rows=[1] * len(f11.data), cols=[1] * len(f11.data))
fig.add_traces(list(f12.data), rows=[1] * len(f12.data), cols=[2] * len(f12.data))

# fig.add_traces(list(f21.data), rows=[2] * len(f21.data), cols=[1] * len(f21.data))
# fig.add_traces(list(f22.data), rows=[2] * len(f22.data), cols=[2] * len(f22.data))

# fig.add_traces(list(f31.data), rows=[3] * len(f31.data), cols=[1] * len(f31.data))
# fig.add_traces(list(f32.data), rows=[3] * len(f32.data), cols=[2] * len(f32.data))
fig.add_traces(list(f31.data), rows=[2] * len(f31.data), cols=[1] * len(f31.data))
fig.add_traces(list(f32.data), rows=[2] * len(f32.data), cols=[2] * len(f32.data))

# --- Synchronise 2D axis ranges ---
# all_2d = [f11, f12, f21, f22]
all_2d = [f11, f12]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f31, f32]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 3):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=800,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:

rows, cols = 2, 4
fig_w, fig_h = 1000, 800

def labels(row, col):
    if col == 1:
        return rf"$f$"
    elif col == 2:
        return rf"$\nabla f$"
    else:
        return ""

overpic_labels(fig, labels, 
            #    stretch_x=0.98, 
               stretch_y=1.35,
               offset_y=25.0)


In [ ]:
data4 = load_image_point_cloud("../data/data4.jpeg", n=1000, threshold=0.95, intensity_weighted=False)
plot_scatter_2d(data4, color = 'black').show()
dg4 = DiffusionGeometry.from_point_cloud(data4)

In [ ]:
f = dg4.function_space.zeros()
f.coeffs[8] = 1
X = f.grad()

vals_1, forms_1 = dg4.laplacian(1).spectrum()
X = forms_1[6].sharp()

f41 = plot_quiver_2d(data4, X.to_ambient(), scale = 0.03, line_width=1.2)
f41.show()

f42 = plot_scatter_2d(data4, color = X.div().to_ambient(), size=4)
f42.show()


In [ ]:
from generate_data import gen_3d_data
data5, _ = gen_3d_data(kind = "torus", minor_radius=1, major_radius=2.5, n = 3000)
dg5 = DiffusionGeometry.from_point_cloud(data5)
camera = dict(eye=dict(x=1., y=1., z=0.7), center=dict(x=0, y=0, z=-0.15))
plot_scatter_3d(data5, color = 'black', camera=camera).show()

In [ ]:
vals_1, forms_1 = dg5.laplacian(1).spectrum()
X = forms_1[7].sharp()

f = dg5.function_space.zeros()
f.coeffs[5] = 1
X = f.grad()

f51 = plot_quiver_3d(data5, X.to_ambient(), scale = 0.1, line_width=2, arrow_scale=1, camera=camera)
f51.update_layout(scene_camera=camera)
f51.show()

f52 = plot_scatter_3d(data5, color = X.div().to_ambient(), camera = camera, size=2.5)
f52.update_layout(scene_camera=camera)
f52.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=2,
    cols=2,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.02,
    row_heights=[0.5, 0.5]
)

# --- Add traces row by row ---
fig.add_traces(list(f41.data), rows=[1] * len(f41.data), cols=[1] * len(f41.data))
fig.add_traces(list(f42.data), rows=[1] * len(f42.data), cols=[2] * len(f42.data))

fig.add_traces(list(f51.data), rows=[2] * len(f51.data), cols=[1] * len(f51.data))
fig.add_traces(list(f52.data), rows=[2] * len(f52.data), cols=[2] * len(f52.data))

# --- Synchronise 2D axis ranges ---
all_2d = [f41, f42]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f51, f52]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 3):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1100,
    height=700,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:

rows, cols = 2, 4
fig_w, fig_h = 1000, 800

def labels(row, col):
    if col == 1:
        return rf"$X$"
    elif col == 2:
       return rf"$\nabla^* X$"
    else:
        return ""

overpic_labels(fig, labels, 
            #    stretch_x=0.98, 
               stretch_y=1,
               offset_y=16.5)


In [ ]:
data6 = load_image_point_cloud("../data/data5.jpeg", n=1200, threshold=0.95, intensity_weighted=False)
plot_scatter_2d(data6, color = 'black').show()
dg6 = DiffusionGeometry.from_point_cloud(data6)

In [ ]:
x, y = data6.T
vortex1 = np.stack((-y-0.2, x+0.5), axis=1)
vortex1 /= np.linalg.norm(vortex1, axis=1, keepdims=True)**2.

vortex2 = np.stack((y+0.3, -x+0.2), axis=1)
vortex2 /= np.linalg.norm(vortex2, axis=1, keepdims=True)**1.8

vortex3 = np.stack((y, -x), axis=1)
vortex3 /= np.linalg.norm(vortex3, axis=1, keepdims=True)**0

vortex = 0.25 * (vortex1 + vortex2 + 2*vortex3)
# vortex = vortex3
a = dg6.form(vortex, 1)

f61 = plot_quiver_2d(data6, a.to_ambient(), scale = 0.05, line_width=1.2)
f61.show()

f62 = plot_2form_2d(data6, a.d().to_ambient())
f62.show()

f63 = plot_scatter_2d(data6, a.d().pointwise_norm(), size=4)
f63.show()

In [ ]:

# Dense cube of points
nx, ny, nz = 10, 10, 10
data7 = np.stack(np.meshgrid(
    np.linspace(-1, 1, nx),
    np.linspace(-1, 1, ny),
    np.linspace(-1, 1, nz),
    indexing='ij'
), axis=-1).reshape(-1, 3)
data7 += 0.02 * np.random.randn(*data7.shape)  # Add slight noise
plot_scatter_3d(data7, color = 'black').show()



In [ ]:


camera7 = dict(eye=dict(x=1.8, y=0.6, z=0.7), center=dict(x=0, y=0, z=-0.2))

x, y, z = data7.T
vortex = np.stack((-y + 0.4*x, x - 0.4*y, 0.*x), axis=1)
vortex /= 1*np.linalg.norm(vortex, axis=1, keepdims=True)**2

dg7 = DiffusionGeometry.from_point_cloud(data7)
a = dg7.form(vortex, 1)

f71 = plot_quiver_3d(data7, a.to_ambient(), scale = 0.1, line_width=2, arrow_scale=0.9)
f71.update_layout(scene_camera=camera7)
f71.show()

f72 = plot_2form_3d(data7, 0.1*a.d().to_ambient())
f72.update_layout(scene_camera=camera7)
f72.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=2,
    cols=2,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.02,
    row_heights=[0.45, 0.55]
)

# --- Add traces row by row ---
fig.add_traces(list(f61.data), rows=[1] * len(f61.data), cols=[1] * len(f61.data))
fig.add_traces(list(f62.data), rows=[1] * len(f62.data), cols=[2] * len(f62.data))

fig.add_traces(list(f71.data), rows=[2] * len(f71.data), cols=[1] * len(f71.data))
fig.add_traces(list(f72.data), rows=[2] * len(f72.data), cols=[2] * len(f72.data))

# --- Synchronise 2D axis ranges ---
all_2d = [f61, f62]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f71, f72]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 3):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera7,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1000,
    height=1000,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:
def labels(row, col):
    if col == 1:
        return rf"$\alpha$"
    elif col == 2:
       return rf"$d \alpha$"
    else:
        return ""
    
overpic_labels(fig, labels, 
            #    stretch_x=0.98, 
               stretch_y=0.96,
               offset_y=21.5)

Codifferential

In [ ]:
data7 = load_image_point_cloud("../data/data6.jpeg", n=800, threshold=0.9, intensity_weighted=True)

plot_scatter_2d(data7, color = 'black').show()
dg7 = DiffusionGeometry.from_point_cloud(data7)

In [ ]:
line_width = 1.

a = dg7.form_space(2).zeros()
a.coeffs[0] = 1

f81 = plot_2form_2d(data7, a.to_ambient())
f81.show()
f91 = plot_quiver_2d(data7, a.codifferential().to_ambient(), scale = 0.01, line_width=line_width)
f91.show()

b = dg7.form_space(2).zeros()
b.coeffs[1] = 1
b.coeffs[4] = -1

f82 = plot_2form_2d(data7, b.to_ambient())
f82.show()
f92 = plot_quiver_2d(data7, b.codifferential().to_ambient(), scale = 0.007, line_width=line_width)
f92.show()

c = dg7.form_space(2).zeros()
c.coeffs[3] = 1

f83 = plot_2form_2d(data7, c.to_ambient())
f83.show()
f93 = plot_quiver_2d(data7, c.codifferential().to_ambient(), scale = 0.01, line_width=line_width)
f93.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.02,
    row_heights=[0.5, 0.5]
)

# --- Add traces row by row ---
fig.add_traces(list(f81.data), rows=[1] * len(f81.data), cols=[1] * len(f81.data))
fig.add_traces(list(f82.data), rows=[1] * len(f82.data), cols=[2] * len(f82.data))
fig.add_traces(list(f83.data), rows=[1] * len(f83.data), cols=[3] * len(f83.data))

fig.add_traces(list(f91.data), rows=[2] * len(f91.data), cols=[1] * len(f91.data))
fig.add_traces(list(f92.data), rows=[2] * len(f92.data), cols=[2] * len(f92.data))
fig.add_traces(list(f93.data), rows=[2] * len(f93.data), cols=[3] * len(f93.data))

# --- Synchronise 2D axis ranges ---
all_2d = [f81, f82, f83, f91, f92, f93]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Final layout ---
fig.update_layout(
    width=1100,
    height=700,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)

clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:
def labels(row, col):
    x = ['alpha', 'beta', 'gamma'][col-1]
    if row == 1:
        return rf"$\{x}$"
    elif row == 2:
       return rf"$d^* \{x}$"
    else:
        return ""
    
overpic_labels(fig, labels, 
            #    stretch_x=0.98, 
               stretch_y=1,
               offset_y=14.)